In [0]:
from pyspark.sql.functions import col, row_number, collect_list,concat, coalesce,concat_ws,format_number,lit,slice
from pyspark.sql.window import Window

# Definindo schema
catalog_name = "cinedata_analytics"
schema_silver = "silver"
schema_gold = "gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_gold}")

In [0]:
#=====================================================================
#                       1° Tabela Dimensao
#=====================================================================

In [0]:
# Referindo a devida tabela a ser trabalhada
df_silver_info = spark.read.table(f"{catalog_name}.{schema_silver}.tb_info_filmes")

df_silver_info = spark.read.table(
    f"{catalog_name}.{schema_silver}.tb_info_filmes"
)

# Criando uma janela para ordenar os dados por id_filme
janela_sk = Window.orderBy("id_filme")

# JUSTIFICATIVA
# A dimensão de filmes utiliza uma chave substituta (sk_movie_id) gerada com row_number(), garantindo uma identificação sequencial independente da chave original. Além disso, id_filme é convertido para STRING, mantendo os demais campos nos tipos definidos pelo esquema.
df_gold_dim_movies = df_silver_info.select(
    row_number().over(janela_sk).cast("bigint").alias("sk_movie_id"),
    col("id_filme").cast("string").alias("id_filme"),
    col("titulo").cast("string").alias("titulo"),
    col("data_lancamento").cast("date").alias("data_lancamento"),
    col("ano_lancamento").cast("int").alias("ano_lancamento"),
    col("duracao_minutos").cast("int").alias("duracao_minutos"),
    col("idioma_original").cast("string").alias("idioma_original"),
    col("status_filme").cast("string").alias("status_filme"),
    col("sinopse").cast("string").alias("sinopse"),
)

# Persistindo a tabela
(
    df_gold_dim_movies.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.{schema_gold}.dim_movies")
)

print("Tabela gold.dim_movies persistida com sucesso!")

In [0]:
#=====================================================================
#                       2° Tabela Dimensao
#=====================================================================

In [0]:
# Referindo a devida tabela a ser trabalhada
df_silver_generos = spark.read.table(
    f"{catalog_name}.{schema_silver}.tb_generos"
)

# Criando uma janela para ordenar os dados por nome_genero
janela_generos = Window.orderBy("nome_genero")

# A dimensão de gêneros utiliza row_number() para gerar uma chave substituta (sk_genre_id) sequencial e independente do dado original. Os gêneros nulos ou vazios são removidos, duplicidades eliminadas e nome_genero mantido como STRING.
df_gold_dim_genres = (
    df_silver_generos.select(
        col("genero").alias("nome_genero"))
        .filter(col("nome_genero").isNotNull() & (col("nome_genero") != ""))
        .distinct()
        .select(
            row_number().over(janela_generos).cast("bigint").alias("sk_genre_id"),
            col("nome_genero").cast("string").alias("nome_genero"),
        )
    )

# Persistindo a tabela   
(
    df_gold_dim_genres.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.{schema_gold}.dim_genres")
)

print("Tabela gold.dim_genres persistida com sucesso!")

In [0]:
#=====================================================================
#                       3° Tabela Dimensao
#=====================================================================

In [0]:
# Referindo a devida tabela a ser trabalhada
df_silver_entidades = spark.read.table(f"{catalog_name}.{schema_silver}.tb_pessoas_empresas")

# Não usamos partitionBy para garantir que o ID é único em toda a tabela.
# Usamos orderBy com as colunas descritivas para garantir ordenação alfabética determinística.
janela_pessoas = Window.orderBy("tipo_pessoa","nome_pessoa")

# Filtragem, deduplicação e geração da Surrogate Key
df_gold_dim_people = (
    df_silver_entidades.filter(
        col("tipo_entidade").isin("Ator", "Diretor", "Roteirista")
    )
    .select(
        col("nome_entidade").alias("nome_pessoa"),
        col("tipo_entidade").alias("tipo_pessoa"),
    )
    .filter(
        col("nome_pessoa").isNotNull()
        & (col("nome_pessoa") != "")
        & (col("nome_pessoa") != "None")
    )
    .distinct()
    .select(
        row_number().over(janela_pessoas).cast("bigint").alias("sk_person_id"),
        col("nome_pessoa").cast("string").alias("nome_pessoa"),
        col("tipo_pessoa").cast("string").alias("tipo_pessoa"),
    )
)

(
    df_gold_dim_people.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.{schema_gold}.dim_people")
)

print(
    f"Tabela {catalog_name}.{schema_gold}.dim_people salva com sucesso com {df_gold_dim_people.count()} registros!"
)

In [0]:
#=====================================================================
#                       4° Tabela Dimensao
#=====================================================================

In [0]:
# Leitura da camada Silver saneada
df_silver_entidades = spark.read.table(
    f"{catalog_name}.{schema_silver}.tb_pessoas_empresas"
)

# Definição da janela determinística para a Surrogate Key ordenada alfabeticamente
janela_produtoras = Window.orderBy("nome_produtora")

# Modelagem dimensional da dim_companies com o nome oficial do contrato
df_gold_dim_companies = (
    df_silver_entidades
    # Escopo de negócio: apenas produtoras/estúdios
    .filter(col("tipo_entidade") == "Produtora")
    # Mapeamento estrito conforme o documento de entrega (nome_produtora)
    .select(col("nome_entidade").alias("nome_produtora"))
    # Integridade contra valores nulos ou vazios
    .filter(
        col("nome_produtora").isNotNull()
        & (col("nome_produtora") != "")
        & (col("nome_produtora") != "None")
    )
    # Granularidade da dimensão: uma linha única por produtora
    .distinct()
    # Atribuição da Surrogate Key e tipagem de contrato
    .select(
        row_number()
        .over(janela_produtoras)
        .cast("bigint")
        .alias("sk_company_id"),
        col("nome_produtora").cast("string").alias("nome_produtora"),
    )
)

# Gravação física no Delta Lake
(
    df_gold_dim_companies.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.{schema_gold}.dim_companies")
)

print(
    f"Tabela {catalog_name}.{schema_gold}.dim_companies persistida com sucesso com {df_gold_dim_companies.count()} registos!"
)

In [0]:
#=====================================================================
#                       5° Tabela Dimensao
#=====================================================================

In [0]:
from pyspark.sql.functions import avg, col, count, round, row_number
from pyspark.sql.window import Window

# 1. Leitura dos dados de origem (Silver de avaliações e Gold de filmes)
df_silver_avaliacoes = spark.read.table(
    f"{catalog_name}.{schema_silver}.tb_avaliacoes_usuarios"
)
df_gold_movies = spark.read.table(f"{catalog_name}.{schema_gold}.dim_movies")

# 2. Agregação das avaliações por filme
# Agrupamos primeiro para reduzir o volume de linhas antes de realizar o join
df_avaliacoes_agrupadas = (
    df_silver_avaliacoes.filter(
        col("id_filme").isNotNull() & col("nota_usuario").isNotNull()
    )
    .groupBy(col("id_filme").cast("string").alias("id_filme"))
    .agg(
        count("nota_usuario").cast("int").alias("qtd_avaliacoes_usuarios"),
        round(avg("nota_usuario"), 2)
        .cast("double")
        .alias("nota_media_usuarios"),
    )
)

# 3. Cruzamento com dim_movies para obter a Surrogate Key sk_movie_id
janela_review = Window.orderBy("sk_movie_id")

df_gold_dim_reviews = (
    df_avaliacoes_agrupadas.join(
        df_gold_movies.select("sk_movie_id", "id_filme"),
        on="id_filme",
        how="inner",
    ).select(
        row_number().over(janela_review).cast("bigint").alias("sk_review_id"),
        col("sk_movie_id").cast("bigint").alias("sk_movie_id"),
        col("qtd_avaliacoes_usuarios")
        .cast("int")
        .alias("qtd_avaliacoes_usuarios"),
        col("nota_media_usuarios").cast("double").alias("nota_media_usuarios"),
    )
)

(
    df_gold_dim_reviews.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.{schema_gold}.dim_reviews")
)

print(
    f"Tabela {catalog_name}.{schema_gold}.dim_reviews persistida com sucesso com {df_gold_dim_reviews.count()} registos!"
)

In [0]:
#=====================================================================
#                       1° Tabela Ponte
#=====================================================================

In [0]:
# Leitura das fontes de dados
df_silver_generos = spark.read.table(
    f"{catalog_name}.{schema_silver}.tb_generos"
)
df_gold_movies = spark.read.table(f"{catalog_name}.{schema_gold}.dim_movies")
df_gold_genres = spark.read.table(f"{catalog_name}.{schema_gold}.dim_genres")

# Modelação da tabela-ponte N:N
df_gold_bridge_movie_genre = (
    df_silver_generos
    # Garante correspondência de tipo no join com a dim_movies (string com string)
    .withColumn("id_filme", col("id_filme").cast("string"))
    # Join 1: Resgata sk_movie_id
    .join(
        df_gold_movies.select("id_filme", "sk_movie_id"),
        on="id_filme",
        how="inner",
    )
    # Join 2: Resgata sk_genre_id ligando genero a nome_genero
    .join(
        df_gold_genres.select("nome_genero", "sk_genre_id"),
        df_silver_generos["genero"] == df_gold_genres["nome_genero"],
        how="inner",
    )
    # Seleção estrita do contrato dimensional
    .select(
        col("sk_movie_id").cast("bigint").alias("sk_movie_id"),
        col("sk_genre_id").cast("bigint").alias("sk_genre_id"),
    )
    # Garante que não haja duplicatas do mesmo par (filme, gênero)
    .distinct()
)

(
    df_gold_bridge_movie_genre.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.{schema_gold}.bridge_movie_genre")
)

print(
    f"Tabela {catalog_name}.{schema_gold}.bridge_movie_genre salva com sucesso com {df_gold_bridge_movie_genre.count()} registos!"
)

In [0]:
#=====================================================================
#                       2° Tabela Ponte
#=====================================================================

In [0]:


# Leitura das fontes necessárias
df_silver_entidades = spark.read.table(
    f"{catalog_name}.{schema_silver}.tb_pessoas_empresas"
)
df_gold_movies = spark.read.table(f"{catalog_name}.{schema_gold}.dim_movies")
df_gold_people = spark.read.table(f"{catalog_name}.{schema_gold}.dim_people")

# Filtragem de pessoas físicas na Silver
df_pessoas_filtradas = df_silver_entidades.filter(
    col("tipo_entidade").isin("Ator", "Diretor", "Roteirista")
).select(
    col("id_filme").cast("string").alias("id_filme"),
    col("nome_entidade").alias("nome_pessoa"),
    col("tipo_entidade").alias("tipo_pessoa"),
)

# Cruzamento para resgatar as Surrogate Keys
df_gold_bridge_movie_person = (
    df_pessoas_filtradas
    # Join 1: Resgata sk_movie_id a partir do id_filme
    .join(
        df_gold_movies.select("id_filme", "sk_movie_id"),
        on="id_filme",
        how="inner",
    )
    # Join 2: Resgata sk_person_id a partir de nome_pessoa E tipo_pessoa
    .join(
        df_gold_people.select("nome_pessoa", "tipo_pessoa", "sk_person_id"),
        on=["nome_pessoa", "tipo_pessoa"],
        how="inner",
    )
    # Seleção dos campos do contrato dimensional
    .select(
        col("sk_movie_id").cast("bigint").alias("sk_movie_id"),
        col("sk_person_id").cast("bigint").alias("sk_person_id"),
    )
    # Garantia de integridade do grão (elimina duplicatas de vínculo)
    .distinct()
)

(
    df_gold_bridge_movie_person.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.{schema_gold}.bridge_movie_person")
)

print(
    f"Tabela {catalog_name}.{schema_gold}.bridge_movie_person persistida com sucesso com {df_gold_bridge_movie_person.count()} registos!"
)

In [0]:
#=====================================================================
#                       3° Tabela Ponte
#=====================================================================

In [0]:
from pyspark.sql.functions import col

# 1. Leitura das tabelas necessárias
df_silver_entidades = spark.read.table(
    f"{catalog_name}.{schema_silver}.tb_pessoas_empresas"
)
df_gold_movies = spark.read.table(f"{catalog_name}.{schema_gold}.dim_movies")
df_gold_companies = spark.read.table(
    f"{catalog_name}.{schema_gold}.dim_companies"
)

# 2. Filtragem e preparação das produtoras na Silver
df_produtoras_silver = (
    df_silver_entidades.filter(col("tipo_entidade") == "Produtora")
    .select(
        col("id_filme").cast("string").alias("id_filme"),
        col("nome_entidade").alias("nome_produtora"),
    )
    .distinct()
)

# 3. Cruzamento para resgatar as Surrogate Keys
df_gold_bridge_movie_company = (
    df_produtoras_silver
    # Join 1: Resgata sk_movie_id a partir do id_filme
    .join(
        df_gold_movies.select("id_filme", "sk_movie_id"),
        on="id_filme",
        how="inner",
    )
    # Join 2: Resgata sk_company_id a partir do nome_produtora
    .join(
        df_gold_companies.select("nome_produtora", "sk_company_id"),
        on="nome_produtora",
        how="inner",
    )
    # Seleção estrita do contrato dimensional
    .select(
        col("sk_movie_id").cast("bigint").alias("sk_movie_id"),
        col("sk_company_id").cast("bigint").alias("sk_company_id"),
    )
    # Garante unicidade do par (filme, produtora)
    .distinct()
)

# Persistência da tabela
(
    df_gold_bridge_movie_company.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.{schema_gold}.bridge_movie_company")
)

print(
    f"Tabela {catalog_name}.{schema_gold}.bridge_movie_company persistida com sucesso com {df_gold_bridge_movie_company.count()} registos!"
)


In [0]:
#=====================================================================
#                        Tabela Fato
#=====================================================================

In [0]:
from pyspark.sql.functions import col

# Leitura das tabelas necessárias
df_gold_movies = spark.read.table(f"{catalog_name}.{schema_gold}.dim_movies")
df_silver_fin = spark.read.table(
    f"{catalog_name}.{schema_silver}.tb_financeiro_filmes"
)
df_silver_eng = spark.read.table(
    f"{catalog_name}.{schema_silver}.tb_metricas_engajamento"
)

# Filtragem dos filmes lançados utilizando o valor correto em português ("Lançado")
df_filmes_lancados = df_gold_movies.filter(
    col("status_filme") == "Lançado"
).select("sk_movie_id", "id_filme")

# Preparação das métricas com alinhamento de tipo (id_filme como string)
df_fin_preparado = df_silver_fin.withColumn(
    "id_filme", col("id_filme").cast("string")
).select(
    "id_filme",
    "orcamento_usd",
    "receita_usd",
    "lucro_usd",
    "orcamento_brl",
    "receita_brl",
    "lucro_brl",
)

df_eng_preparado = df_silver_eng.withColumn(
    "id_filme", col("id_filme").cast("string")
).select(
    "id_filme",
    "popularidade",
    "nota_media_tmdb",
    "qtd_votos_tmdb",
    "nota_media_imdb",
    "qtd_votos_imdb",
)

# Cruzamento preservando estritamente o grão 1:1 por filme lançado
df_gold_fact_performance = (
    df_filmes_lancados.join(df_fin_preparado, on="id_filme", how="left")
    .join(df_eng_preparado, on="id_filme", how="left")
    # Tipagem estrita de acordo com o contrato oficial da Entrega 1
    .select(
        col("sk_movie_id").cast("bigint").alias("sk_movie_id"),
        # Métricas Financeiras (DECIMAL 18,2)
        col("orcamento_usd").cast("decimal(18,2)").alias("orcamento_usd"),
        col("receita_usd").cast("decimal(18,2)").alias("receita_usd"),
        col("lucro_usd").cast("decimal(18,2)").alias("lucro_usd"),
        col("orcamento_brl").cast("decimal(18,2)").alias("orcamento_brl"),
        col("receita_brl").cast("decimal(18,2)").alias("receita_brl"),
        col("lucro_brl").cast("decimal(18,2)").alias("lucro_brl"),
        # Métricas de Engajamento
        col("popularidade").cast("double").alias("popularidade"),
        col("nota_media_tmdb").cast("double").alias("nota_media_tmdb"),
        col("qtd_votos_tmdb").cast("int").alias("qtd_votos_tmdb"),
        col("nota_media_imdb").cast("double").alias("nota_media_imdb"),
        col("qtd_votos_imdb").cast("int").alias("qtd_votos_imdb"),
    )
)

(
    df_gold_fact_performance.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.{schema_gold}.fact_movies_performance")
)

print(
    f"Tabela {catalog_name}.{schema_gold}.fact_movies_performance gravada com sucesso com {df_gold_fact_performance.count()} registos!"
)

In [0]:
#=====================================================================
#                          Tabela IA
#=====================================================================

In [0]:
# Leitura das tabelas Gold consolidadas
df_dim_movies = spark.read.table(f"{catalog_name}.{schema_gold}.dim_movies")
df_fact_perf = spark.read.table(
    f"{catalog_name}.{schema_gold}.fact_movies_performance"
)
df_dim_people = spark.read.table(f"{catalog_name}.{schema_gold}.dim_people")
df_bridge_person = spark.read.table(
    f"{catalog_name}.{schema_gold}.bridge_movie_person"
)

# Agregação de Atores Principais (limite de até 5 atores para síntese de contexto)
df_atores = (
    df_bridge_person.join(df_dim_people, on="sk_person_id", how="inner")
    .filter(col("tipo_pessoa") == "Ator")
    .groupBy("sk_movie_id")
    .agg(
        concat_ws(
            ", ", slice(collect_list("nome_pessoa"), 1, 5)
        ).alias(  # type: ignore
            "atores_principais"
        )
    )
)

# Agregação de Diretores
df_diretores = (
    df_bridge_person.join(df_dim_people, on="sk_person_id", how="inner")
    .filter(col("tipo_pessoa") == "Diretor")
    .groupBy("sk_movie_id")
    .agg(
        concat_ws(", ", collect_list("nome_pessoa")).alias("diretor")  # type: ignore
    )
)

# Cruzamento consolidado e construção do documento textual com fallbacks
df_context_base = (
    df_dim_movies.join(df_fact_perf, on="sk_movie_id", how="inner")
    .join(df_atores, on="sk_movie_id", how="left")
    .join(df_diretores, on="sk_movie_id", how="left")
)
 
df_gold_genai_movies_context = df_context_base.select(
    col("id_filme").cast("string").alias("movie_id"),
    col("titulo").cast("string").alias("title"),
    concat(
        lit("O filme "),
        coalesce(col("titulo"), lit("Sem Título")),
        lit(", lançado no ano de "),
        coalesce(col("ano_lancamento").cast("string"), lit("ano não informado")),
        lit(", faturou "),
        coalesce(
            concat(lit("$"), format_number(col("receita_usd"), 2)),
            lit("um valor não divulgado"),
        ),
        lit(" e teve um custo de "),
        coalesce(
            concat(lit("$"), format_number(col("orcamento_usd"), 2)),
            lit("um valor não divulgado"),
        ),
        lit(". Estrelado por "),
        coalesce(col("atores_principais"), lit("elenco não informado")),
        lit(" e dirigido por "),
        coalesce(col("diretor"), lit("diretor não informado")),
        lit(", o filme possui a seguinte sinopse: "),
        coalesce(col("sinopse"), lit("Sinopse não disponível.")),
    )
    .cast("string")
    .alias("llm_context_document"),
)

(
    df_gold_genai_movies_context.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.{schema_gold}.gold_genai_movies_context")
)

print(
    f"Tabela {catalog_name}.{schema_gold}.gold_genai_movies_context persistida com sucesso com {df_gold_genai_movies_context.count()} registos!"
)